[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/b_28_kv_cache_pure_solution.ipynb)

# 🔴 Solution: KV Cache Attention without Flax

*Attention & Transformers · Hard*

Reference implementation. Try it yourself in `b_28_kv_cache_pure.ipynb` first.

---
Problem 14 with no Flax — and a cache that was always a value, not state.

### Signature
```python
class KVCacheAttention:
    def __init__(self, d_model, num_heads, *, key): ...
    def __call__(self, x, cache=None): ...     # -> (out, (k_all, v_all))
```

| | shape |
|---|---|
| `x` | `(B, seq_new, d_model)` |
| `cache` | `None`, or `(k, v)` each `(B, H, seq_past, d_k)` |
| `out` | `(B, seq_new, d_model)` |
| returned cache | `(k, v)` each `(B, H, seq_past + seq_new, d_k)` |

Same four projections as `b_26`, from `jax.random.split(key, 4)`.

### The cache never needed a module
In problem 14 it already had to be returned rather than mutated, because JAX
arrays are immutable — so `nnx.Module` was buying you nothing there. Written as
a plain class that becomes obvious: the cache goes in as an argument and comes
back as a return value.

### The mask is the hard part, and it is silent
Query `i` of this chunk sits at absolute position `seq_past + i`, so it may
attend to keys `0 … seq_past + i`:

$$j - i \le \text{seq\_past}
\quad\Longrightarrow\quad
\texttt{tril(ones((seq\_new, seq\_total)), k=seq\_total - seq\_new)}$$

Forget the `k=` and you get the top-left triangle, which hides every cached
key. Nothing errors — the model just stops seeing its own history. During
single-token decode `seq_new == 1` and the mask is all-True, so the bug only
appears when you prefill more than one token at a time.

### The test that catches everything
Run a sequence two ways — all at once, versus prefill-then-decode-one-at-a-time
— and require they agree. A wrong mask offset, a wrong concat axis, or
re-projecting the cache all fail it.

### Why this exists alongside problem 14
Interview sandboxes often ship `jax` alone. Same class name, same arguments,
same `W_q`/`W_k`/`W_v`/`W_o` attributes as the `nnx` version, so practising it
reinforces problem 14 rather than competing with it. `Linear` is handed to you
the way `nnx.Linear` is.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp


class Linear:
    """Given to you, exactly as nnx.Linear is given to you in problem 14."""

    def __init__(self, d_in, d_out, *, key):
        self.kernel = jax.random.normal(key, (d_in, d_out)) / jnp.sqrt(d_in)
        self.bias = jnp.zeros((d_out,))

    def __call__(self, x):
        return x @ self.kernel + self.bias


class KVCacheAttention:
    def __init__(self, d_model, num_heads, *, key):
        self.h = num_heads
        self.d_k = d_model // num_heads
        kq, kk, kv, ko = jax.random.split(key, 4)
        self.W_q = Linear(d_model, d_model, key=kq)
        self.W_k = Linear(d_model, d_model, key=kk)
        self.W_v = Linear(d_model, d_model, key=kv)
        self.W_o = Linear(d_model, d_model, key=ko)

    def __call__(self, x, cache=None):
        split = lambda t: t.reshape(*t.shape[:-1], self.h, self.d_k).swapaxes(-3, -2)
        q, k, v = split(self.W_q(x)), split(self.W_k(x)), split(self.W_v(x))

        # A value we extend, not state we mutate. The cached keys are already
        # projected, so they go in untouched.
        if cache is not None:
            k = jnp.concatenate([cache[0], k], axis=-2)
            v = jnp.concatenate([cache[1], v], axis=-2)

        s = jnp.einsum("...hqd,...hkd->...hqk", q, k) / jnp.sqrt(
            jnp.asarray(self.d_k, q.dtype)
        )
        seq_new, seq_total = s.shape[-2], s.shape[-1]
        # Query i is at absolute position seq_past + i, so j - i <= seq_past.
        # Plain tril would hide the entire cache.
        allowed = jnp.tril(
            jnp.ones((seq_new, seq_total), dtype=bool), k=seq_total - seq_new
        )
        o = jnp.einsum(
            "...hqk,...hkd->...hqd", jax.nn.softmax(jnp.where(allowed, s, -jnp.inf), axis=-1), v
        )

        o = o.swapaxes(-3, -2)
        return self.W_o(o.reshape(*o.shape[:-2], self.h * self.d_k)), (k, v)

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp

attn = KVCacheAttention(8, 2, key=jax.random.key(0))
x = jax.random.normal(jax.random.key(1), (1, 8, 8))

full, _ = attn(x)

out, cache = attn(x[:, :5])
print("prefill 5:", out.shape, " cache:", cache[0].shape)
pieces = [out]
for t in range(5, 8):
    out, cache = attn(x[:, t:t + 1], cache)
    print(f"  decode {t}: cache grew to {cache[0].shape[-2]}")
    pieces.append(out)

step = jnp.concatenate(pieces, axis=1)
print("\nstepwise == all-at-once?", bool(jnp.allclose(step, full, atol=1e-5)),
      f"(max diff {float(jnp.max(jnp.abs(step - full))):.2e})")

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("kv_cache_pure")